In [192]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [193]:
import warnings
warnings.filterwarnings('ignore')

In [194]:
df = pd.read_csv('data/preprocessed_data.csv')

In [195]:
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

In [196]:
from sklearn.model_selection import train_test_split

In [197]:
from sklearn.preprocessing import StandardScaler

In [198]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [199]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [200]:
lr = LogisticRegression(max_iter=1000, random_state=42)
svm = SVC(probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)

SOFT VOTING

In [201]:
voting_clf = VotingClassifier(
    estimators=[('lr', lr), ('svm', svm), ('rf', rf), ('mlp', mlp)],
    voting='soft'
)

In [202]:
voting_clf.fit(X_train_scaled, y_train)
y_pred = voting_clf.predict(X_test_scaled)

In [203]:
accuracy_score(y_test, y_pred)

0.9824561403508771

In [204]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99        72
           1       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



STACKING

In [205]:
from sklearn.ensemble import StackingClassifier

In [206]:
stacking_clf = StackingClassifier(
    estimators=[('lr', lr), ('svm', svm), ('rf', rf), ('mlp', mlp)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    stack_method='predict_proba'
)

In [207]:
stacking_clf.fit(X_train_scaled, y_train)
y_pred = stacking_clf.predict(X_test_scaled)

In [208]:
accuracy_score(y_test, y_pred)

0.9824561403508771

In [209]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99        72
           1       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [210]:
gb = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb.fit(X_train_scaled, y_train)
y_pred = gb.predict(X_test_scaled)

In [211]:
accuracy_score(y_test, y_pred)

0.9649122807017544

In [212]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114



Required comparison

In [213]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix

In [214]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [215]:
raw_lr = LogisticRegression(max_iter=1000)
raw_rf = RandomForestClassifier()
raw_svc = SVC(probability=True)

In [216]:
num_feature = X.select_dtypes(exclude='object').columns

In [217]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

In [218]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_feature)
    ]
)

In [219]:
clean_lr = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

clean_rf = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier())
])

clean_svc = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(probability=True))
])

In [220]:
from sklearn.ensemble import VotingClassifier

ensemble = Pipeline([
    ("preprocess", preprocessor),
    ("model", VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("rf", RandomForestClassifier()),
            ("svc", SVC(probability=True))
        ],
        voting="soft"
    ))
])

In [221]:
def evaluate(true, pred, proba=None):
    
    AccuracyScore = accuracy_score(true, pred)
    f1Score = f1_score(true, pred)
    recallScore = recall_score(true, pred)
    confusionMatrix = confusion_matrix(true, pred)
    
    roc_auc = None
    if proba is not None:
        roc_auc = roc_auc_score(true, proba)
    
    return AccuracyScore, f1Score, recallScore, roc_auc, confusionMatrix

In [222]:
models = {
    "RAW_LR": raw_lr,
    "RAW_RF": raw_rf,
    "RAW_SVC": raw_svc,
    "CLEAN_LR": clean_lr,
    "CLEAN_RF": clean_rf,
    "CLEAN_SVC": clean_svc,
    "ENSEMBLE": ensemble
}

In [223]:
results = {}

for name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = None
    
    results[name] = evaluate(y_test, y_pred, y_proba)

In [224]:
summary = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [results[m][0] for m in results],
    "F1(Malignant)": [results[m][1] for m in results],
    "Recall(Malignant)": [results[m][2] for m in results],
    "ROC-AUC": [results[m][3] for m in results],
})

print(summary.sort_values(by="Recall(Malignant)", ascending=False))

       Model  Accuracy  F1(Malignant)  Recall(Malignant)   ROC-AUC
6   ENSEMBLE  0.982456       0.975610           0.952381  0.996032
3   CLEAN_LR  0.964912       0.951220           0.928571  0.995370
1     RAW_RF  0.973684       0.962963           0.928571  0.995866
5  CLEAN_SVC  0.973684       0.962963           0.928571  0.995040
4   CLEAN_RF  0.964912       0.950000           0.904762  0.996032
0     RAW_LR  0.929825       0.897436           0.833333  0.992725
2    RAW_SVC  0.921053       0.880000           0.785714  0.982143


In [225]:
for name in results:
    
    acc, f1, recall, roc, cm = results[name]
    
    print("\n======================")
    print(name)
    print("Confusion Matrix:")
    print(cm)


RAW_LR
Confusion Matrix:
[[71  1]
 [ 7 35]]

RAW_RF
Confusion Matrix:
[[72  0]
 [ 3 39]]

RAW_SVC
Confusion Matrix:
[[72  0]
 [ 9 33]]

CLEAN_LR
Confusion Matrix:
[[71  1]
 [ 3 39]]

CLEAN_RF
Confusion Matrix:
[[72  0]
 [ 4 38]]

CLEAN_SVC
Confusion Matrix:
[[72  0]
 [ 3 39]]

ENSEMBLE
Confusion Matrix:
[[72  0]
 [ 2 40]]
